# Nephron - Stage 1 model (preliminary screening, no laboratory values)

This notebook follows CRISP-DM. It trains **Random Forest** and **XGBoost** on the preliminary-screening questions of `ml/feature_catalog.json`,
compares them, isotonic-calibrates the winner (XGBoost), and saves that calibrated pipeline (the project's production model) together with
`model_metadata.json`, exactly as the website needs them.

## How to use it
1. Open the **Settings** cell and check `PROJECT_DIR` (the Flask project). Usually that is the only thing to change.
2. Run **Kernel > Restart and Run All**. A full run on 400,000 rows takes a while (expect somewhere between 10 and 30 minutes, depending on the laptop). To try the notebook quickly first, set `SAMPLE_ROWS_FOR_A_QUICK_TRIAL = 50000`. A quick trial saves a model too, so run the full notebook again before you use the result.
3. Open a terminal in the Flask project and run `python verify_model.py`. Every line must say PASS. Then restart the website.

The notebook never edits the dataset. It writes only `ml/model/final_model.joblib`, `ml/model/model_metadata.json` and `ml/validation_holdout.csv`.

## Read this before you present the results
* **The data is synthetic.** A questionnaire alone predicting CKD very well (see the ROC-AUC below) is a property of how the data was generated. It is **not** evidence that it works on real patients. Say so.
* **About half of the rows have CKD.** The predicted probability is the probability *in a population like this dataset*, not a person's absolute risk in the general population.
  Nephron therefore calls it "Predicted Risk Probability" and never presents it as a diagnosis or a real-world risk.
* **`Country` carries no signal** (the CKD rate is the same in every country), and the data has no Philippines category. A person in the Philippines chooses "Other". This is a finding to discuss with your adviser.
* **In this file, people who do not drink alcohol are blank**, not the text `None`. The notebook makes that explicit (one assumption, printed below). Please confirm it with whoever produced the dataset.

## 1. Business understanding
* **Goal:** an early **screening** tool. It estimates the probability of the high-risk class (CKD = 1) from questions a person can answer without a laboratory. It never diagnoses.
* **Inputs:** the 28 Stage 1 predictors (no laboratory value, no `Frailty_Index`). Obesity is worked out from the BMI, as the dataset defines it.
* **Models:** only Random Forest and XGBoost. XGBoost is the production model; Random Forest is the comparison. If Random Forest is clearly better, the notebook says so, and that is a decision for the team.
* **Success:** good discrimination on data the model never saw, sensible calibration, a validated threshold that belongs to the saved model, and a file the website can verify.

## 2. Settings and environment

In [ ]:
# ======================= SETTINGS: the only cell you may need to edit =======================
from pathlib import Path

PROJECT_DIR = Path(r"D:\4ADA\NEPHRON_DEVELOPMENT\SYSTEM_DEV")    # the Flask project (it contains app.py and the ml folder)
DATA_PATH = None                     # for example Path(r"D:\...\CKD_400k.csv"). None = look for CKD_400k.csv near this notebook and the project
MODEL_DIR = PROJECT_DIR / "ml" / "model"      # where the website looks for the model. Do not change it.
FILE_NAME = "final_model.joblib"

TARGET = "CKD"                       # 1 = CKD (the high-risk class), 0 = healthy
HIGH_RISK_CLASS = 1
RANDOM_STATE = 42
SAMPLE_ROWS_FOR_A_QUICK_TRIAL = None # for example 50000 to test the notebook quickly. Leave None for the real run.
TUNING_ROWS = 60_000                 # rows used to search hyperparameters (the final models are trained on all training rows)

In [ ]:
import datetime as dt
import hashlib
import json
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

try:
    import xgboost
    xgboost_version = version("xgboost")
except (ImportError, PackageNotFoundError):
    raise SystemExit("XGBoost is not installed in this environment. Run:  pip install xgboost   then restart the kernel.")

from packaging.version import Version
print("Python       ", sys.version.split()[0])
print("scikit-learn ", sklearn.__version__)
print("xgboost      ", xgboost_version)
print("pandas       ", pd.__version__)
print("numpy        ", np.__version__)

assert Version(sklearn.__version__) >= Version("1.2"), "scikit-learn 1.2 or newer is needed. Run: pip install -U scikit-learn"
if Version(sklearn.__version__) >= Version("1.6") and Version(xgboost_version) < Version("2.1.4"):
    raise SystemExit("This xgboost is too old for this scikit-learn. Run:  pip install -U xgboost  then restart the kernel.")
assert (PROJECT_DIR / "ml" / "feature_catalog.json").exists(), f"{PROJECT_DIR} is not the Flask project (ml/feature_catalog.json is missing). Fix PROJECT_DIR in the Settings cell."
print("\nThe Flask project was found:", PROJECT_DIR)

## 3. Data understanding
Pandas normally turns the words `None`, `NA` and `null` into missing values. Here we read the file so that **only an empty cell** is missing.

In [ ]:
def find_data_file():
    if DATA_PATH is not None:
        return Path(DATA_PATH)
    here = Path.cwd()
    places = [here, here.parent, here.parent.parent, PROJECT_DIR, PROJECT_DIR.parent,
              PROJECT_DIR.parent / "NEPHRON_ML", PROJECT_DIR.parent / "NEPHRON_ML" / "model"]
    for folder in places:
        if (folder / "CKD_400k.csv").exists():
            return folder / "CKD_400k.csv"
    raise FileNotFoundError("CKD_400k.csv was not found near this notebook or the project. "
                            "Set DATA_PATH in the Settings cell, for example DATA_PATH = Path(r'D:\\path\\to\\CKD_400k.csv').")

data_file = find_data_file()
raw = pd.read_csv(data_file, keep_default_na=False, na_values=[""])
print("File:", data_file)
print(f"{len(raw):,} rows x {raw.shape[1]} columns | Patient_ID unique: {raw['Patient_ID'].is_unique} | exact duplicate rows: {int(raw.duplicated().sum())}")
missing = raw.isna().sum()
print("Columns with missing (empty) cells:", {c: int(n) for c, n in missing[missing > 0].items()})

In [ ]:
# The target
assert set(raw[TARGET].unique()) == {0, 1}, "the target must be 0 and 1"
print(f"{TARGET}: {int(raw[TARGET].sum()):,} of {len(raw):,} rows have CKD ({raw[TARGET].mean():.1%}).")
print("\nStage among the rows with CKD:")
print(raw.loc[raw[TARGET] == 1, "CKD_Stage"].value_counts(normalize=True).round(3).to_string())
print("\nCKD rate by age band:")
print(raw.groupby(pd.cut(raw["Age"], [17, 40, 60, 75, 95]), observed=True)[TARGET].mean().round(3).to_string())
print("\nCKD rate by country (if it is the same everywhere, Country carries no signal):")
print(raw.groupby("Country")[TARGET].mean().round(3).to_string())

### Three assumptions: what a blank answer means
In the earlier dataset the people who do not drink were written as the word `None`. In this file the same group is **blank**, and the same is
true for two of the diet questions added later. If we left these blank the model pipeline would silently fill them in with the most common
answer instead. So each is made an explicit answer, and what was changed is counted and printed.

In [ ]:
def recode_blank(column, label, becomes):
    blank = raw[column].isna()
    print(f"{column}: {int(blank.sum()):,} rows ({blank.mean():.1%}) are blank. Recoded to the answer '{becomes}' ({label}).")
    raw[column] = raw[column].fillna(becomes)

recode_blank("Alcohol_Consumption", "does not drink", "None")
print("  (in the earlier dataset the answer 'None' was 35.9% of the rows)")
recode_blank("SugaryDrinksAddedSugar", "no sugar-sweetened drinks", "None")
recode_blank("UltraProcessedFoodFrequency", "never eats ultra-processed food", "Never")
print("\nAlcohol_Consumption now:   ", raw["Alcohol_Consumption"].value_counts().to_dict())
print("SugaryDrinksAddedSugar now:", raw["SugaryDrinksAddedSugar"].value_counts().to_dict())
print("UltraProcessedFoodFrequency now:", raw["UltraProcessedFoodFrequency"].value_counts().to_dict())

## 4. Data preparation
### 4.1 The screening questions (the contract with the website)
The list of predictors is **not typed here**. It is read from `ml/feature_catalog.json`, the same file the website's form and checks use.
Every question is compared with the data, so a model can only be trained on inputs the form can really ask, with answers the model has really seen.

In [ ]:
catalog = json.load(open(PROJECT_DIR / "ml" / "feature_catalog.json", encoding="utf-8"))
QUESTIONS = catalog["questions"]
FEATURES = [q["name"] for q in QUESTIONS]                                   # the model inputs, in the catalog's order
CATEGORICAL = [q["name"] for q in QUESTIONS if q["kind"] == "choice"]        # answers that are words
NUMERIC = [q["name"] for q in QUESTIONS if q["kind"] != "choice"]            # numbers, yes/no (0/1) and the derived Obesity
print(f"{len(FEATURES)} inputs: {len(CATEGORICAL)} categorical + {len(NUMERIC)} numeric (including yes/no)")

problems = []
for q in QUESTIONS:
    name, kind = q["name"], q["kind"]
    if name not in raw.columns:
        problems.append(f"{name}: this column is not in the dataset")
        continue
    column = raw[name]
    if column.isna().any():
        problems.append(f"{name}: {int(column.isna().sum())} missing values (the model must be trained on complete answers)")
    if kind == "choice":
        have, want = set(column.astype(str).unique()), {o["value"] for o in q["options"]}
        if have != want:
            problems.append(f"{name}: the data has {sorted(have)} but the form offers {sorted(want)}")
    elif kind in ("yesno", "derived"):
        if not set(column.dropna().unique()) <= {0, 1}:
            problems.append(f"{name}: expected 0 and 1 but found {sorted(column.dropna().unique())[:6]}")
    elif kind in ("number", "bmi"):
        if q["min"] < column.min() or q["max"] > column.max():
            problems.append(f"{name}: the form accepts {q['min']} to {q['max']} but the data only covers {column.min()} to {column.max()}")
if problems:
    raise ValueError("The catalog and the dataset disagree:\n  - " + "\n  - ".join(problems))
print("Every screening question matches the training data (answers offered, 0/1 answers, and number ranges).")

# The two derived facts the website relies on
assert (raw["Obesity"] == (raw["BMI"] >= 30).astype(int)).all(), "Obesity is not BMI >= 30 in this data"
bmi_from_height_weight = (raw["Weight_kg"] / (raw["Height_cm"] / 100) ** 2).round(1)
assert (bmi_from_height_weight == raw["BMI"]).mean() > 0.999, "BMI is not weight / height^2 in this data"
print("Obesity = (BMI >= 30) and BMI = weight / height^2, for every row. The website computes both from height and weight.")

In [ ]:
# ---- What is NOT used, and why (an audit, so nobody has to wonder) ----
LABORATORY_AND_VITALS = ["Systolic_BP", "Diastolic_BP", "Blood_Pressure_Category", "Serum_Creatinine", "eGFR", "Blood_Urea_Nitrogen", "Albumin",
                         "Urine_ACR", "Albuminuria_Category", "Urine_Protein", "HbA1c", "Fasting_Glucose", "Hemoglobin", "Sodium", "Potassium",
                         "Calcium", "Phosphorus", "Uric_Acid", "Total_Cholesterol", "HDL", "LDL", "Triglycerides", "CRP"]
OUTCOMES_AND_IDS = ["Patient_ID", "CKD", "CKD_Stage", "Kidney_Failure_Risk", "Dialysis_Required", "Hospitalization_Risk"]
FRAILTY = ["Frailty_Index", "Frailty_Category"]
banned = set(LABORATORY_AND_VITALS) | set(OUTCOMES_AND_IDS) | set(FRAILTY)
assert not (set(FEATURES) & banned), "a laboratory value, an outcome or a frailty column is among the predictors"
assert TARGET not in FEATURES
others = [c for c in raw.columns if c not in FEATURES and c not in banned]
print(f"Used as predictors: {len(FEATURES)}")
print(f"Not used, laboratory or vitals (a preliminary screening has no lab): {len(LABORATORY_AND_VITALS)}")
print(f"Not used, outcomes or identifiers (would leak the answer): {len(OUTCOMES_AND_IDS) - 1}, plus the target itself")
print(f"Not used, Frailty (removed by decision): {len(FRAILTY)}")
print(f"Not used, other columns of this dataset that are not part of Stage 1: {len(others)}")

### 4.2 Three separate sets: train (70%), validation (15%) and test (15%)
The **validation** set picks the threshold. The **test** set is looked at once, at the end.

In [ ]:
from sklearn.model_selection import train_test_split

data = raw
if SAMPLE_ROWS_FOR_A_QUICK_TRIAL:
    data = raw.sample(SAMPLE_ROWS_FOR_A_QUICK_TRIAL, random_state=RANDOM_STATE)
    print(f"QUICK TRIAL: training on only {len(data):,} of {len(raw):,} rows. The result is NOT the real model. Run the notebook again in full before using it.\n")
X = data[FEATURES].copy()                   # the columns are in the catalog's order, so the model's input order is known
y = data[TARGET].astype(int)
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_hold, y_hold, test_size=0.50, stratify=y_hold, random_state=RANDOM_STATE)
for label, part in (("train", y_train), ("validation", y_val), ("test", y_test)):
    print(f"{label:11s} {len(part):>8,} rows | CKD rate {part.mean():.3f}")

### 4.3 Preprocessing inside the pipeline
* Words become 0/1 columns with `OneHotEncoder(handle_unknown="error")`: an answer the model never saw makes the pipeline **refuse**, instead of quietly treating it as another answer.
* There is **no imputer** on purpose. A missing answer must never be filled in for a person. The website refuses incomplete answers before the model is asked.
* Numbers and yes/no answers pass through unchanged.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def new_preprocessor():
    return ColumnTransformer(
        [("categorical", OneHotEncoder(handle_unknown="error", sparse_output=False), CATEGORICAL),
         ("numeric", "passthrough", NUMERIC)],
        sparse_threshold=0.0,
    )
print("Preprocessing ready.")

## 5. Modeling
Both models sit in the same kind of pipeline. Hyperparameters are searched on a sample of the training rows (3-fold cross-validation, ROC-AUC); the final models are then trained on **all** training rows.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

rf_pipeline = Pipeline([("preprocess", new_preprocessor()),
                        ("model", RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))])
xgb_pipeline = Pipeline([("preprocess", new_preprocessor()),
                         ("model", XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE, verbosity=0))])

rf_space = {"model__n_estimators": [100, 150], "model__max_depth": [8, 12, 16], "model__min_samples_leaf": [20, 50, 100],
            "model__max_features": ["sqrt", 0.5]}
xgb_space = {"model__n_estimators": [200, 400, 600], "model__max_depth": [3, 4, 5, 6], "model__learning_rate": [0.03, 0.05, 0.1],
             "model__subsample": [0.7, 0.85, 1.0], "model__colsample_bytree": [0.6, 0.8, 1.0],
             "model__min_child_weight": [1, 5, 20], "model__reg_lambda": [1, 5, 10]}

tune_rows = min(TUNING_ROWS, len(X_train))
X_tune = X_train.sample(tune_rows, random_state=RANDOM_STATE)
y_tune = y_train.loc[X_tune.index]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print(f"Searching hyperparameters on {tune_rows:,} training rows...")

In [ ]:
rf_search = RandomizedSearchCV(rf_pipeline, rf_space, n_iter=6, scoring="roc_auc", cv=cv, random_state=RANDOM_STATE, n_jobs=1)
rf_search.fit(X_tune, y_tune)
print("Random Forest best cross-validated ROC-AUC: %.4f" % rf_search.best_score_)
print("  ", rf_search.best_params_)

In [ ]:
xgb_search = RandomizedSearchCV(xgb_pipeline, xgb_space, n_iter=15, scoring="roc_auc", cv=cv, random_state=RANDOM_STATE, n_jobs=1)
xgb_search.fit(X_tune, y_tune)
print("XGBoost best cross-validated ROC-AUC: %.4f" % xgb_search.best_score_)
print("  ", xgb_search.best_params_)

In [ ]:
# The final models: the best settings, trained on ALL training rows (and only the training rows).
rf_final = clone(rf_pipeline).set_params(**rf_search.best_params_).fit(X_train, y_train)
xgb_final = clone(xgb_pipeline).set_params(**xgb_search.best_params_).fit(X_train, y_train)
print("Both final models are trained on", f"{len(X_train):,}", "rows.")
print("Classes the XGBoost model knows:", list(xgb_final.steps[-1][1].classes_))

## 6. Evaluation
The probability of the high-risk class is always taken **by class label** from `classes_`, never assumed to be the second column.

In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score, roc_curve

def high_probability(fitted_pipeline, X_part):
    # classes_ is read from the OUTER fitted object, which is correct whether it is a plain Pipeline or a
    # CalibratedClassifierCV wrapping one - both expose it the same way, and it is what predict_proba's own
    # columns are actually indexed by.
    classes = list(fitted_pipeline.classes_)
    return fitted_pipeline.predict_proba(X_part)[:, classes.index(HIGH_RISK_CLASS)]

def youden(y_true, probabilities):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    best = int(np.argmax(tpr - fpr))               # Youden's J = sensitivity + specificity - 1. roc_curve means "score >= threshold".
    return float(thresholds[best]), float(tpr[best]), float(1 - fpr[best])

p_val = {"Random Forest": high_probability(rf_final, X_val), "XGBoost": high_probability(xgb_final, X_val)}
table = pd.DataFrame({name: {"ROC-AUC": roc_auc_score(y_val, p), "PR-AUC": average_precision_score(y_val, p), "Brier score (lower is better)": brier_score_loss(y_val, p)}
                      for name, p in p_val.items()}).T.round(4)
print("VALIDATION set (the models never trained on it):")
print(table.to_string())
gap = table.loc["Random Forest", "ROC-AUC"] - table.loc["XGBoost", "ROC-AUC"]
if gap > 0.01:
    print(f"\nNOTE: Random Forest is better than XGBoost by {gap:.3f} ROC-AUC. Nephron's production model is XGBoost by decision; discuss this with your adviser.")
else:
    print("\nXGBoost is as good as Random Forest (within 0.01 ROC-AUC), so the decision to use XGBoost costs nothing.")

In [ ]:
# The threshold is chosen on the VALIDATION set, for the model that will be saved (XGBoost, trained on the training rows only).
threshold, sensitivity_val, specificity_val = youden(y_val, p_val["XGBoost"])
print(f"Youden's J threshold for the XGBoost model: {threshold:.4f}  (validation sensitivity {sensitivity_val:.3f}, specificity {specificity_val:.3f})")
print(f"Rule used by the website: probability >= {threshold:.4f} means High Preliminary Risk.")

In [ ]:
# The TEST set, looked at once, at the threshold chosen on validation.
p_test = high_probability(xgb_final, X_test)
predicted_high = p_test >= threshold
tn, fp, fn, tp = confusion_matrix(y_test, predicted_high).ravel()
test_metrics = {
    "ROC-AUC": roc_auc_score(y_test, p_test), "PR-AUC": average_precision_score(y_test, p_test), "Brier": brier_score_loss(y_test, p_test),
    "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp), "PPV (precision)": tp / (tp + fp), "NPV": tn / (tn + fn),
    "accuracy": (tp + tn) / len(y_test),
}
print(f"TEST set, {len(y_test):,} rows, XGBoost at the validated threshold:")
for name, value in test_metrics.items():
    print(f"  {name:16s} {value:.4f}")
print(f"\nConfusion matrix  (rows: truth, columns: predicted)\n                 predicted Low   predicted High\n  truly no CKD   {tn:>13,}   {fp:>14,}\n  truly CKD      {fn:>13,}   {tp:>14,}")
print(f"\nMean predicted probability {p_test.mean():.3f} vs observed CKD rate {y_test.mean():.3f}  (the probability is relative to a dataset in which about {y_train.mean():.0%} have CKD)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for name, fitted in (("Random Forest", rf_final), ("XGBoost", xgb_final)):
    fpr, tpr, _ = roc_curve(y_test, high_probability(fitted, X_test))
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, high_probability(fitted, X_test)):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey"); axes[0].set_xlabel("1 - specificity"); axes[0].set_ylabel("sensitivity"); axes[0].set_title("ROC curve (test set)"); axes[0].legend()

bins = pd.qcut(pd.Series(p_test), 10, duplicates="drop")
calib = pd.DataFrame({"predicted": p_test, "observed": y_test.to_numpy()}).groupby(bins.to_numpy(), observed=True).mean()
axes[1].plot(calib["predicted"], calib["observed"], "o-", label="XGBoost"); axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("mean predicted probability"); axes[1].set_ylabel("observed CKD rate"); axes[1].set_title("Calibration (test set, deciles)"); axes[1].legend()

def importance_by_question(fitted_pipeline):
    names = list(fitted_pipeline[:-1].get_feature_names_out())
    weights = fitted_pipeline.steps[-1][1].feature_importances_
    totals = dict.fromkeys(FEATURES, 0.0)
    for encoded, weight in zip(names, weights):
        base = encoded.split("__", 1)[1]
        owner = max([f for f in FEATURES if base == f or base.startswith(f + "_")], key=len)     # Diabetes_Medication is not Diabetes
        totals[owner] += float(weight)
    total = sum(totals.values())
    return pd.Series({k: v / total for k, v in totals.items()}).sort_values(ascending=False)

importance = importance_by_question(xgb_final)
importance.head(10)[::-1].plot.barh(ax=axes[2], color="#7D75EE"); axes[2].set_title("What XGBoost relies on most (share of importance)")
plt.tight_layout(); plt.show()
print("Top factors overall:", ", ".join(f"{k} ({v:.1%})" for k, v in importance.head(6).items()))
print("Country's share:", f"{importance['Country']:.2%}", "(if this is tiny, the Country question adds nothing to the model)")

In [ ]:
# Does it work about equally well for different groups? (test set, ROC-AUC per group)
def auc_by(groups, title):
    rows = {}
    for group, index in X_test.groupby(groups, observed=True).groups.items():
        if y_test.loc[index].nunique() == 2 and len(index) >= 200:
            rows[str(group)] = (len(index), roc_auc_score(y_test.loc[index], p_test[X_test.index.get_indexer(index)]))
    print(f"\n{title}:"); print(pd.DataFrame(rows, index=["rows", "ROC-AUC"]).T.round(3).to_string())

auc_by(X_test["Sex"], "By sex"); auc_by(X_test["Ethnicity"], "By ethnicity"); auc_by(X_test["Country"], "By country")
auc_by(pd.cut(X_test["Age"], [17, 40, 60, 75, 95]), "By age band")

In [ ]:
# The pipeline itself refuses an answer it has never seen (the website also refuses it before asking the model).
one_person = X_test.iloc[[0]].copy()
one_person["Country"] = "Philippines"
try:
    xgb_final.predict_proba(one_person)
    print("!!! The pipeline accepted an unknown country. Do not use this model.")
except ValueError as error:
    print("Refused, as it should be:", str(error).splitlines()[0][:110])

## 6.1 Calibrating the chosen model
A raw model\'s probability is a good RANKING but not always a good PROBABILITY: rows it scores near 70% might not actually
have a 70% observed rate of CKD. Isotonic regression fixes this with a monotonic curve fitted on OUT-OF-FOLD predictions,
so the model is never calibrated on rows it was tested against. `CalibratedClassifierCV(method="isotonic", ensemble=False)`
does exactly this: it cross-validates internally (5 folds) to fit the curve, then refits the XGBoost pipeline itself on
ALL of the training rows (`ensemble=False` keeps this to one fitted model and one curve, not several averaged together).
Only the WINNING model (XGBoost) is calibrated and saved; Random Forest was only needed for the comparison above.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

xgb_calibrated = CalibratedClassifierCV(clone(xgb_pipeline).set_params(**xgb_search.best_params_), method="isotonic", cv=5, ensemble=False)
xgb_calibrated.fit(X_train, y_train)
assert len(xgb_calibrated.calibrated_classifiers_) == 1                 # ensemble=False: exactly one fitted XGBoost, one calibration curve

p_val_raw, p_val_calibrated = high_probability(xgb_final, X_val), high_probability(xgb_calibrated, X_val)
reliability = pd.DataFrame({"predicted (raw)": p_val_raw, "predicted (calibrated)": p_val_calibrated, "actually CKD": y_val.to_numpy()})
bins = pd.qcut(p_val_raw, 10, duplicates="drop")
print("Validation set, by decile of the RAW score - does calibration bring the predicted probability closer to what was actually observed?")
print(reliability.groupby(bins.to_numpy(), observed=True).mean().round(3).to_string())
print(f"\nBrier score, validation set (lower is better): raw {brier_score_loss(y_val, p_val_raw):.4f}  |  calibrated {brier_score_loss(y_val, p_val_calibrated):.4f}")

## 7. Deployment: save the model for the website
The threshold below is computed **from the very (calibrated) model being saved**, on the validation set it never trained on, so it
cannot belong to another model. The checks at the end read the file back and compare it with the model in memory.

In [ ]:
# The saved object is CalibratedClassifierCV wrapping Pipeline[ColumnTransformer -> XGBoost], not a bare Pipeline:
# unwrap it the same way the website\'s services/model_service.py does, to check the REAL model underneath.
calibrated_classifiers = xgb_calibrated.calibrated_classifiers_
assert len(calibrated_classifiers) == 1, "ensemble=False must give exactly one fitted model and one calibration curve"
inner_pipeline = calibrated_classifiers[0].estimator
model = inner_pipeline.steps[-1][1]
assert type(model).__name__ == "XGBClassifier", "Nephron's production model is XGBoost"
assert hasattr(xgb_calibrated, "feature_names_in_"), "the pipeline must be trained on a pandas DataFrame"
assert [str(c) for c in xgb_calibrated.feature_names_in_] == FEATURES
assert list(model.classes_) == list(xgb_calibrated.classes_), "the inner XGBoost and the outer calibrated wrapper must agree on the class order"
classes = list(xgb_calibrated.classes_)
assert HIGH_RISK_CLASS in classes, f"classes are {classes}"

p_high_val = high_probability(xgb_calibrated, X_val)             # the CALIBRATED probability, found by class label
final_threshold, _, _ = youden(y_val, p_high_val)                 # Youden's J on the validation set, for THIS (calibrated) model
assert 0 < final_threshold < 1
print(f"Youden's J threshold, uncalibrated model (section 6, for comparison only): {threshold:.4f}")
print(f"Youden's J threshold, calibrated model (this is the one saved):           {final_threshold:.4f}")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = MODEL_DIR / FILE_NAME
joblib.dump(xgb_calibrated, artifact_path)
sha256 = hashlib.sha256(artifact_path.read_bytes()).hexdigest()

def plain(value):
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return float(value)
    return value

metadata = {
    "schema_version": 1,
    "artifact_file": FILE_NAME,
    "artifact_sha256": sha256,
    "model_name": "XGBoost",
    "model_version": f"xgboost-stage1-{dt.date.today():%Y%m%d}-{sha256[:8]}" + ("-QUICKTRIAL" if SAMPLE_ROWS_FOR_A_QUICK_TRIAL else ""),
    "high_risk_class": HIGH_RISK_CLASS,
    "threshold": final_threshold,
    "threshold_method": f"Youden's J on the isotonic-calibrated probability of the high-risk class, validation split ({len(y_val):,} rows)",
    "calibration": {"method": "isotonic", "cv_folds": 5, "ensemble": False,
                    "note": "fitted on out-of-fold predictions of the training set; the base XGBoost is then refit on all of it"},
    "stage": "preliminary",
    "features": [str(c) for c in xgb_calibrated.feature_names_in_],
    "library_versions": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "xgboost": xgboost_version,
                         "pandas": pd.__version__, "numpy": np.__version__},
    # Extra facts for the panel (the website ignores them)
    "trained_on": data_file.name,
    "quick_trial": bool(SAMPLE_ROWS_FOR_A_QUICK_TRIAL),
    "rows_train_validation_test": [len(y_train), len(y_val), len(y_test)],
    "ckd_rate_in_training_data": round(float(y_train.mean()), 4),
    "test_roc_auc": round(float(test_metrics["ROC-AUC"]), 4),
    "hyperparameters": {k.replace("model__", ""): plain(v) for k, v in xgb_search.best_params_.items()},
    "note": "Synthetic data. The probability is relative to a dataset in which about half of the people have CKD. Screening, not diagnosis.",
}
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", artifact_path)
print("Saved:", MODEL_DIR / "model_metadata.json")
print(f"Model version {metadata['model_version']} | threshold {final_threshold:.6f} | {len(metadata['features'])} inputs")
if SAMPLE_ROWS_FOR_A_QUICK_TRIAL:
    print("\n*** This was a QUICK TRIAL. The saved model is labelled -QUICKTRIAL. Set SAMPLE_ROWS_FOR_A_QUICK_TRIAL = None and run the notebook again in full. ***")

In [ ]:
# ---- Read it back and prove it is the same model ----
loaded = joblib.load(artifact_path)
sample = X_test.iloc[:3000]
assert np.allclose(high_probability(loaded, sample), high_probability(xgb_calibrated, sample)), "the saved file does not give the same probabilities"
assert hashlib.sha256(artifact_path.read_bytes()).hexdigest() == metadata["artifact_sha256"]
print("The saved file gives exactly the same probabilities as the model in memory, and its checksum matches the metadata.")

# ---- A held-out file so verify_model.py can prove the threshold belongs to this model ----
validation_file = PROJECT_DIR / "ml" / "validation_holdout.csv"
pd.concat([X_val, y_val], axis=1).to_csv(validation_file, index=False)          # 'None' is written as the word None
print("Saved:", validation_file, f"({len(X_val):,} validation rows, never used for training)")

### Next
Open a terminal in the Flask project and run these two commands. **Every line must say PASS / ACCEPTED**:

```
cd D:\4ADA\NEPHRON_DEVELOPMENT\SYSTEM_DEV
python verify_model.py
python verify_model.py --validation-csv ml\validation_holdout.csv --target CKD
```

The Flask project must use the **same library versions** as this notebook (they were printed at the top and saved in `model_metadata.json`).
If `verify_model.py` says they differ, it prints the exact `pip install` command.
Then restart the website. If everything passes, the dashboard's **Start Assessment** opens.

`ml/validation_holdout.csv` is a data file. Do not put it in a public repository.